# PolyglotSound : une seule cellule

1. Menu *Execution > Modifier le type d'execution > **GPU T4***.
2. Reglez l'extrait ci-dessous (20 s = rapide), puis cliquez sur **Executer** (ou *Execution > Tout executer*).
3. Premiere fois : 10 a 15 min d'installation. Ensuite un lien **https://xxxx.gradio.live** s'affiche : ouvrez-le.
4. Dans le studio : importez le morceau, *Analyser*, onglet 2, corrigez les paroles, **Creer la nouvelle chanson**, ecoutez.

Licences : NLLB-200 et MMS-TTS non commerciaux, Seed-VC GPL-3.0. Ne convertissez jamais la voix de quelqu'un sans son accord ecrit.

In [ ]:
#@title PolyglotSound : installer et lancer { display-mode: "form" }
DEBUT_EXTRAIT_S = 0  #@param {type:"number"}
DUREE_EXTRAIT_S = 20  #@param {type:"number"}
IA_LOCALE_OLLAMA = True  #@param {type:"boolean"}
VOIX_SEED_VC = True  #@param {type:"boolean"}
import os, subprocess
os.chdir('/content')
if not os.path.exists('/content/polyglotsound'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Souraka229/polyglotsound'], check=True)
else:
    subprocess.run(['git', '-C', '/content/polyglotsound', 'pull', '-q'], check=False)
os.chdir('/content/polyglotsound')
if not os.path.exists('.venv/.polyglot_ok'):
    env = dict(os.environ, WITH_LLM='1' if IA_LOCALE_OLLAMA else '0', WITH_SEEDVC='1' if VOIX_SEED_VC else '0')
    r = subprocess.run(r'set -o pipefail; bash scripts/setup_colab.sh 2>&1 | grep -v -E "Downloading|Downloaded|Prepared|Resolved|Audited|^ \+ "',
                       shell=True, env=env, executable='/bin/bash')
    if r.returncode != 0:
        raise SystemExit('Installation echouee : lisez le message ci-dessus, puis relancez la cellule.')
    open('.venv/.polyglot_ok', 'w').write('ok')
if IA_LOCALE_OLLAMA:
    subprocess.run('pgrep -x ollama >/dev/null || (nohup ollama serve > /tmp/ollama.log 2>&1 &)', shell=True)
os.environ['MPLBACKEND'] = 'Agg'  # Colab force un backend inline absent du venv
os.environ['POLYGLOT_EXCERPT_START'] = str(DEBUT_EXTRAIT_S)
os.environ['POLYGLOT_EXCERPT_DUR'] = str(DUREE_EXTRAIT_S)
print('Studio en cours de demarrage : cliquez sur le lien https://....gradio.live ci-dessous')
!.venv/bin/python -m app.ui.gradio_app --share --host 0.0.0.0

### En cas de souci
- *Session redemarree* : relancez la cellule (Colab efface `/content` a chaque nouvelle session : reinstallation).
- *Pas de lien gradio.live* : relancez la cellule (le service de partage peut etre momentanement indisponible).
- *Fon* : Whisper ne reconnait pas le fon, tapez les paroles a l'onglet 2.
- *Tests* : `!cd /content/polyglotsound && .venv/bin/python -m pytest -q`